### Read historical price data, resample to 15min and merge with the load data

In [ ]:
import pandas as pd 
import numpy as np


price_hist_file = pd.ExcelFile(r"raw_input\Historical_Wholesale_Prices.xlsx")

da_price = []
for sheet in price_hist_file.sheet_names:
    df = price_hist_file.parse(sheet, index_col=0)
    df.index = df.index.rename('datetime')
    df.index = pd.to_datetime(df.index).tz_localize('UTC').tz_convert('Europe/Berlin')
    df = df.filter(like='[EUR/MWh]')
    df = df.dropna()
    df.columns = df.columns.str.replace(' Price', '').str.replace(' [EUR/MWh]', '_price', regex=False).str.lower()
    
    if df.index.diff()[1].total_seconds() / 60 != 15: 
        df = df.resample("15min").ffill()
    if df.columns[0].startswith('da'):
        da_price.append(df)

    
da_price = pd.concat(da_price, axis=0).sort_index()
prices_hist = da_price.join(df, how='outer').sort_index()
prices_hist = prices_hist.ffill()

prices_hist.to_pickle("input/historical_prices.pkl")

### Read historical daily averages, add daily price stats and time-based features, merge into a training dataset


In [24]:
import holidays

hist_file = pd.ExcelFile(r"raw_input\Historical_Fundamentals.xlsx")

hist_dfs = []

for sheet in hist_file.sheet_names:
    var_name = sheet.replace('Mean ', '').replace(' ', '_').lower()
    var_name = var_name.replace('_generation', '').replace('_load', '_demand')
    df = hist_file.parse(sheet, index_col=0)
    df.index = df.index.rename('datetime')
    df.index = pd.to_datetime(df.index).tz_localize('Europe/Berlin')
    hist = df.filter(like="Actual").iloc[:,0].rename(var_name).dropna()
    hist_dfs.append(hist)


hist_df = pd.concat(hist_dfs, axis=1)
hist_df = hist_df.dropna()

## add daily day-ahead prices statistic price stats to the dataset
da_stats = prices_hist['da_price'].resample('D').agg(['max', 'min', 'mean', 'std'])
da_stats.columns = ['da_price_' + col for col in da_stats.columns]

X_train =  da_stats.join(hist_df, how='outer').dropna()
# add time-based features
X_train["weekend"] = X_train.index.dayofweek.isin([5, 6]) * 1
X_train["holiday"] = X_train.index.map(lambda x: x in holidays.Germany()) * 1
X_train["quarter"] = X_train.index.quarter
X_train.index = X_train.index.rename("datetime")

# save the dataset for later use in the forecasting notebook
X_train.to_pickle(r"input\historical_data.pkl")


### Read forecast hourly values, add daily price stats and time-based features, merge into a forecasting dataset

In [ ]:
forecast_file = pd.ExcelFile(r"raw_input\Forecasts.xlsx")

forecast_dfs = []

for sheet in forecast_file.sheet_names:
    var_name = sheet.replace(' Forecast', '').replace(' ', '_').lower()
    df = forecast_file.parse(sheet, index_col=0)
    df.index = df.index.rename('datetime')
    df.index = pd.to_datetime(df.index).tz_localize('Europe/Berlin')
    actual = df.filter(like="Actual").iloc[:,0].rename(var_name)
    forecast = df.filter(like="Forecast").iloc[:,0].rename(var_name)
    
    # Solar, wind and load forecasts have a "Normal" column that can be used to fill missing values in the forecast.
    if df.columns.str.contains("Normal").any():
        normal = df.filter(like="Normal").iloc[:,0].rename(var_name)
        forecast = forecast.fillna(normal)
    
    forecast_dfs.append(actual.fillna(forecast))

forecast_df = pd.concat(forecast_dfs, axis=1)
forecast_df = forecast_df.dropna()
forecast_df = forecast_df.rename(columns={"da": "da_price"})

forecast_start = all_hist_data.index.max().round('H') 
forecast_df = forecast_df.loc[forecast_start:,]


C:\Users\c.fusarbassini\AppData\Local\Temp\2\ipykernel_13172\3334542947.py:24: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  forecast_start = all_hist_data.index.max().round('H')
